# Notebook 2: RAG Pipeline with OpenSearch + Ollama

In this notebook you will:
1. Run **semantic (k-NN) search** against the indexed podcast transcripts
2. Run **hybrid search** combining vector similarity + BM25 keyword matching
3. Build a complete **RAG pipeline**: query → retrieve → generate answer with Ollama
4. Compare results and explore interesting podcast content

---
### Prerequisites
- Notebook 1 completed (podcasts indexed)
- Virtual env active (`source .venv/bin/activate`)
- `OLLAMA_MODEL` set in your environment:
  ```bash
  export OLLAMA_MODEL=gemma4
  ```

## 0 · Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

from sentence_transformers import SentenceTransformer
from src.opensearch_client import get_client, knn_search, hybrid_search, INDEX_NAME
from src.rag import ask, ask_streaming, build_context
from dotenv import load_dotenv
import ollama

load_dotenv()

client = get_client()
model = SentenceTransformer('all-MiniLM-L6-v2')  # loads instantly from cache

count = client.count(index=INDEX_NAME)['count']
print(f"✅ OpenSearch ready: {count} documents indexed")

if len(ollama.list().models) <1:
    print("⚠️  OLLAMA_MODEL not set and loaded. Semantic search works, but RAG won't generate answers")
else:
    print(f"✅ Ollama configured with {ollama.list().models[0].model} loaded")


def embed(text: str) -> list[float]:
    return model.encode(text, normalize_embeddings=True).tolist()

✅ OpenSearch ready: 1018 documents indexed
✅ Ollama configured with gemma4:latest loaded


## 1 · Semantic Search (pure k-NN)

We embed the query and find the most similar transcript chunks using **HNSW approximate nearest neighbour** search.
No keywords need to match, the similarity is purely semantic.

In [2]:
query = "How does HNSW approximate nearest neighbor search work?"

results = knn_search(client, embed(query), k=5)

print(f"Query: '{query}'\n")
print("─" * 70)
for i, r in enumerate(results, 1):
    print(f"[{i}] Score: {r['score']:.4f}")
    print(f"    Episode: {r['episode_title']}")
    print(f"    Text:    {r['chunk_text'][:300]}")
    print()

Query: 'How does HNSW approximate nearest neighbor search work?'

──────────────────────────────────────────────────────────────────────
[1] Score: 0.6799
    Episode: Joan Fontanals - Principal Engineer - Jina AI
    Text:    also python? I don't know if we are for instance I think we are also using some bindings for HLW so you are using probably C++ version of HNSW binding to python right? yes that's for sure but I don't know if some of for the HNSW you part yes for some other parts I don't know we are so maybe sites an

[2] Score: 0.6730
    Episode: Filip Haltmayer (Data Engineer, Ziliz) on Milvus vector database and working with clients
    Text:    seeing what's actually going on behind the scenes in the box. But yeah, these two might pop out with two completely different values. They might be in completely different clusters, even though they kind of should be similar. So that's where this, it can kind of go wrong. And then, see, yeah, you se

[3] Score: 0.6557
    Episode: Yury

In [3]:
# Try a few more, notice that semantically related content surfaces even without exact keyword matches
queries = [
    "What makes vector databases different from traditional databases?",
    "Challenges of embedding computation at scale",
    "How to evaluate search quality and relevance?",
]

for q in queries:
    results = knn_search(client, embed(q), k=1)
    print(f"Q: {q}")
    print(f"   → [{results[0]['score']:.3f}] {results[0]['episode_title']}")
    print(f"      {results[0]['chunk_text'][:150]}...")
    print()

Q: What makes vector databases different from traditional databases?
   → [0.748] Vector Databases: The Rise, Fall and Future - by NotebookLM
      articles about vector databases. Oh, really? Huh? I wonder why. What do you make of that? Well, you know, it kind of hints at a potential shift in the...

Q: Challenges of embedding computation at scale
   → [0.672] Trey Grainger - Wormhole Vectors
      generate a new vector. You can average vectors together to find, you know, sort of an area that's in between them. But there's lots of mathematical op...

Q: How to evaluate search quality and relevance?
   → [0.686] Tom Lackner - VP Engineering - Classic.com - on Qdrant, NFT, challenges and joys of ML engineering
      attention to the title. So like if these words occur in the title, they will rank the document higher. And at that point, I was like, this is like mag...



## 2 · Hybrid Search (k-NN + BM25)

Hybrid search combines two signals:
- **Vector similarity** → captures *meaning*  
- **BM25 keyword match** → captures *exact terminology*

This is especially useful when queries contain specific technical terms (like `HNSW`, `Pinecone`, `Qdrant`) that may not be well-represented in the embedding space.

In [4]:
query = "Qdrant filtering and payload indexing strategies"

print("=" * 70)
print("SEMANTIC ONLY (k-NN)")
print("=" * 70)
semantic_results = knn_search(client, embed(query), k=3)
for r in semantic_results:
    print(f"  [{r['score']:.4f}] {r['episode_title'][:60]}")

print()
print("=" * 70)
print("HYBRID (k-NN + BM25)")
print("=" * 70)
hybrid_results = hybrid_search(client, query, embed(query), k=3)
for r in hybrid_results:
    print(f"  [{r['score']:.4f}] {r['episode_title'][:60]}")

SEMANTIC ONLY (k-NN)
  [0.6335] Tom Lackner - VP Engineering - Classic.com - on Qdrant, NFT,
  [0.6282] Yury Malkov - Staff Engineer, Twitter - Author of the most a
  [0.6281] Yury Malkov - Staff Engineer, Twitter - Author of the most a

HYBRID (k-NN + BM25)
  [2.8894] Sid Probstein - Creator of SWIRL - Search in siloed data wit
  [2.8809] Yury Malkov - Staff Engineer, Twitter - Author of the most a
  [2.7261] Yury Malkov - Staff Engineer, Twitter - Author of the most a


## 3 · The RAG Pipeline

```md
User Question
      │
      ▼
  [Embed query]          ← sentence-transformers
      │
      ▼
  [Retrieve chunks]      ← OpenSearch k-NN / hybrid search
      │
      ▼
  [Build context]        ← format top-k results into a prompt
      │
      ▼
  [Generate answer]      ← Ollama (local LLM, e.g. llama3.2)
      │
      ▼
   Answer + Citations
```

In [29]:
def rag_query(question: str, k: int = 5, search_mode: str = "hybrid", stream: bool = True):
    """Full RAG pipeline: embed → retrieve → generate."""
    query_vec = embed(question)

    if search_mode == "hybrid":
        hits = hybrid_search(client, question, query_vec, k=k)
    else:
        hits = knn_search(client, query_vec, k=k)

    print(f"Question: {question}")
    print(f"Retrieved {len(hits)} chunks via {search_mode} search")
    print("Sources:", ", ".join(set(h['episode_title'] for h in hits)))
    print("\n" + "─" * 70)
    print("Answer:")
    print()

    if stream:
        ask_streaming(question, hits)
    else:
        answer = ask(question, hits)
        print(answer)

    return hits

In [30]:
# Question 1: Technical deep-dive
hits = rag_query(
    "What is HNSW and why did Yury Malkov invent it? What problems does it solve?"
)

Question: What is HNSW and why did Yury Malkov invent it? What problems does it solve?
Retrieved 5 chunks via hybrid search
Sources: Jo Bergum - Distinguished Engineer, Yahoo! Vespa - Journey of Vespa from Sparse into Neural Search, Debunking myths of vector search and LLMs with Leo Boytsov, Yury Malkov - Staff Engineer, Twitter - Author of the most adopted ANN algorithm HNSW

──────────────────────────────────────────────────────────────────────
Answer:

HNSW (the algorithm/library) is described as:

*   A highly popular and best-in-class algorithm used in vector search (Yury Malkov - Staff Engineer, Twitter; Debunking myths of vector search and LLMs with Leo Boytsov).
*   A core algorithm in the field of vector search (Debunking myths of vector search and LLMs with Leo Boytsov).

**Regarding its invention and purpose:**

*   The author, Yuri Malkov, worked on a startup that was building distributed scalable search systems, and the resulting work led to several papers on the predecess

In [5]:
# Question 2: Cross-episode synthesis
hits = rag_query(
    "What are the main differences between Pinecone, Weaviate, Qdrant, and Milvus "
    "as discussed by podcast guests?"
)

Question: What are the main differences between Pinecone, Weaviate, Qdrant, and Milvus as discussed by podcast guests?
Retrieved 5 chunks via hybrid search
Sources: Yusuf Sarıgöz - AI Research Engineer, Qdrant - Getting to know your data with metric learning, Bob van Luijt (CEO, Semi) on the Weaviate vector search engine, Filip Haltmayer (Data Engineer, Ziliz) on Milvus vector database and working with clients, Connor Shorten - PhD Researcher - Florida Atlantic University & Founder at Henry AI Labs, Max Irwin - Founder, MAX.IO - On economics of scale in embedding computation with Mighty

──────────────────────────────────────────────────────────────────────
Answer:

Based only on the provided context, the main differences in approach and language support between the vector databases are:

*   **Weaviate:**
    *   **Architecture/Flexibility:** It is praised for its great job of having different clients for different languages and stacks (Episode: Max Irwin - Founder, MAX.IO - On econom

In [6]:
# Question 3: Practical advice
hits = rag_query(
    "What advice do guests give for deploying vector search in production at scale?"
)

Question: What advice do guests give for deploying vector search in production at scale?
Retrieved 5 chunks via hybrid search
Sources: Yaniv Vaknin - Director of Product, Searchium - Hardware accelerated vector search, Trey Grainger - Wormhole Vectors, Yusuf Sarıgöz - AI Research Engineer, Qdrant - Getting to know your data with metric learning

──────────────────────────────────────────────────────────────────────
Answer:

According to the podcast guests, the advice for deploying metric learning and vector search in production at scale includes:

*   Putting your metric learning model into production.
*   Combining vector search with "super search" (filtering) to allow for filtering your data.

(Episode: Yusuf Sarıgöz - AI Research Engineer, Qdrant - Getting to know your data with metric learning)


In [7]:
# Question 4: Hybrid search concepts
hits = rag_query(
    "How do wormhole vectors work and how are they different from standard hybrid search?"
)

Question: How do wormhole vectors work and how are they different from standard hybrid search?
Retrieved 5 chunks via hybrid search
Sources: Trey Grainger - Wormhole Vectors

──────────────────────────────────────────────────────────────────────
Answer:

**How Wormhole Vectors Work**

According to the podcast guests, the concept of wormhole vectors is an emerging way to improve query understanding and recall by leveraging multiple vector spaces:

*   **General Process:** A query is run, finding relevant documents which are then used as a "wormhole." A "wormhole vector" is generated to "hop" from the original vector space to a corresponding meaning or region in a different, target vector space (Trey Grainger - Wormhole Vectors).
*   **Creating the Vector:** To move to a dense vector space, the vector can be created by:
    *   Pulling the vectors of the top $N$ documents.
    *   Averaging the vectors of the top $N$ documents.
    *   For example, if a keyword search is performed and to

## 4 · Inspect Retrieved Context

It's important to understand what the LLM sees. Let's inspect the context window.

In [8]:
query = "What are the economics of running vector search at scale?"
query_vec = embed(query)
hits = hybrid_search(client, query, query_vec, k=5)

context = build_context(hits)
print(f"Context sent to LLM ({len(context)} chars):")
print("─" * 70)
print(context)

Context sent to LLM (4520 chars):
──────────────────────────────────────────────────────────────────────
[1] Episode: Max Irwin - Founder, MAX.IO - On economics of scale in embedding computation with Mighty
    Score: 3.3981
    Text: you you still have to do fine tuning for most cases but you're not going to start fine tuning before you even know how this thing performs like. So in the beginning right you want to try a model and see what how close it is so there's some starting starting work there I know Algolia is getting to the vector search stuff so I don't know maybe they maybe they don't know how to choose a model so you guys you can use my default model if you want it's just. Yeah absolutely I mean so far what I hear from you is that mighty has the qualities let's say it can run on pure CPU which is a win on cost it scales which is also a win on cost in the long term right and it also is insanely fast which is a win on product it's a win on go to market like I have this document

## 5 · Your Turn 🫵 Try Your Own Questions

Explore the 33 episodes by asking your own questions!

In [11]:
# What topics are in the podcast? Let's list all episodes
import json
resp = client.search(
    index=INDEX_NAME,
    body={
        "size": 0,
        "aggs": {
            "episodes": {
                "terms": {"field": "episode_title", "size": 50}
            }
        }
    }
)
print("All indexed episodes:")
for b in resp['aggregations']['episodes']['buckets']:
    print(f"  [{b['doc_count']:3d} chunks] {b['key']}")

All indexed episodes:
  [ 54 chunks] Max Irwin - Founder, MAX.IO - On economics of scale in embedding computation with Mighty
  [ 48 chunks] Connor Shorten - Research Scientist, Weaviate - ChatGPT, LLMs, Form vs Meaning
  [ 47 chunks] Atita Arora - Search Relevance Consultant - Revolutionizing E-commerce with Vector Search
  [ 47 chunks] Bob van Luijt (CEO, Semi) on the Weaviate vector search engine
  [ 46 chunks] Sid Probstein - Creator of SWIRL - Search in siloed data with LLMs
  [ 44 chunks] Doug Turnbull - Staff Relevance Engineer, Shopify - Search as a constant experimentation cycle
  [ 40 chunks] Economical way of serving vector search workloads with Simon Eskildsen, CEO Turbopuffer
  [ 40 chunks] Filip Haltmayer (Data Engineer, Ziliz) on Milvus vector database and working with clients
  [ 40 chunks] Jo Bergum - Distinguished Engineer, Yahoo! Vespa - Journey of Vespa from Sparse into Neural Search
  [ 38 chunks] Evgeniya Sukhodolskaya - Data Advocate, Toloka - Data at the core of

In [9]:
# Your question here!
my_question = "What is the difference between dense and sparse vectors?"

rag_query(my_question)

Question: What is the difference between dense and sparse vectors?
Retrieved 5 chunks via hybrid search
Sources: Debunking myths of vector search and LLMs with Leo Boytsov, Trey Grainger - Wormhole Vectors

──────────────────────────────────────────────────────────────────────
Answer:

According to the podcast guests, the difference between dense and sparse vectors can be summarized as follows:

*   **Sparse Vectors:**
    *   These are associated with the "bag of words representations" (Debunking myths of vector search and LLMs with Leo Boytsov).
    *   A document is represented by a sparse vector that can contain either zeros and ones (indicating if specific terms are present or not) or weights (Debunking myths of vector search and LLMs with Leo Boytsov).

*   **Dense Vectors:**
    *   These methods involve representing text using fixed size vectors, which became possible with the development of deep learning (Debunking myths of vector search and LLMs with Leo Boytsov).
    *   The

[{'score': 4.551123,
  'episode_title': 'Debunking myths of vector search and LLMs with Leo Boytsov',
  'chunk_text': "model, why don't you represent each document using both sparse and dense vector? And when you're computing the similarity, you can compute the similarity between sparse parts, between dense parts, and then combine them somehow. For example, using a weight. And that's in fact what I was trying to do in my thesis as well, because I was doing, my similarities was basically an ensemble of several similarities course for at least two representations. And that could work. There's of course modern instantiations of this, and there's a paper, I think both are by some glue people, where they did exactly like this, they combined splaid and some dense vector embeddings. And that can work apparently a little bit better than, or sometimes maybe a lot better than basic representations, like each representation specifically. So with both approaches, of course, there are issues that y

## 6 · Key Takeaways

| Concept | What we built |
|---|---|
| **Vector Index** | OpenSearch k-NN index with HNSW + cosine similarity |
| **Embeddings** | `all-MiniLM-L6-v2` 384 dims, fast CPU inference |
| **Semantic Search** | k-NN finds conceptually related chunks regardless of exact keywords |
| **Hybrid Search** | BM25 + k-NN combined, better for specific terminology |
| **RAG** | Retrieved context grounds the LLM's answers in real podcast content |

### What to explore next

| Notebook | Topic |
|---|---|
| **03** | Cross-encoder **re-ranking:** re-score top-k for higher precision |
| **04** | **Metadata filtering:** scope search by episode title or date range |
| **05** | **Larger embeddings:** swap to `all-mpnet-base-v2` (768 dims) |
| **06** | **Aiven OpenSearch:** managed cloud cluster with TLS |
| **07** | **Streamlit UI:** streaming chat app with source citations |